# 3Di alignment benchmark

## Description

We compare the performance of ESM3Di, compared to ProstT5, on the MetaVR sequence set.

We have 3 different 3Di representations of the MetaVR sequence set:
* `AF3`: 3Di generated by Foldseek based on the predicted AlphaFold3 structures of the MetaVR sequence set.
* `ProstT5`: 3Di generated by ProstT5 based on the sequences of the MetaVR sequence set.
* `ESM3Di`: 3Di generated by ESM3Di based on the sequences of the MetaVR sequence set.

We compile each representation into a Foldseek database, run Foldseek `structurealign` module with 1-vs-1 basis (aligning identical entries).

The results are represented as the bitscore of the alignment, and 3Di sequence identity, which can be calculated as: 
$\text{Identity} = \frac{\text{\# of residues with identical 3Di}}{\text{\# of residues}}$

## Database preparation

### Download and process the MetaVR dataset

In [ ]:
%%bash
# Download MetaVR dataset
mkdir -p data/metavr
wget -q -O data/metavr/IMGVR5_PC_3Dmodels.tar.gz "https://www.meta-virome.org/Data/Downloads/IMGVR5_PC_3Dmodels.tar.gz"
tar -xzf data/metavr/IMGVR5_PC_3Dmodels.tar.gz -C data/metavr

# Create Foldseek database
mkdir -p data/foldseek
foldseek createdb data/metavr/metaVR_structures_all data/foldseek/metavr_af3

### Predict 3Di representations with ProstT5

#### Generate AA FASTA file from the MetaVR dataset

In [4]:
%%bash
awk '{gsub(/\x00/, "", $0)} NR==FNR{f[FNR]=$0; next} f[FNR]{print ">"f[FNR]"\n"$0}' \
    data/foldseek/metavr_af3_h \
    data/foldseek/metavr_af3 \
    > data/metavr/metavr_aa.fa

#### Predict 3Di with ProstT5

In [ ]:
%%bash
foldseek databases ProstT5 data/prostt5_weights data/tmp
rm -rf data/tmp
foldseek createdb data/metavr/metavr_aa.fa data/foldseek/metavr_prostt5 --gpu 1 --prostt5-model data/prostt5_weights